# ⚡ Element-wise Operations Benchmark: ROCm vs CPU

Element-wise tensor operations (`add`, `mul`, `exp`, `sin`, `relu`) are embarrassingly parallel.
On large tensors the GPU can saturate its cores with independent work, minimising the
impact of HIP kernel dispatch overhead.

In [ ]:
import torch
from metalcheck import device_info, is_rocm_available
from metalcheck.utils import benchmark_elementwise

## 1. Device & System Information

In [ ]:
info = device_info()
for k, v in info.items():
    label = k.replace("_", " ").title()
    print(f"{label:<20s}: {v}")

## 2. Element-wise Benchmark

Benchmark five common element-wise ops at increasing tensor sizes (1 M to 100 M elements).

In [ ]:
import matplotlib.pyplot as plt
import seaborn as sns
import pandas as pd

sns.set_theme(style="whitegrid")

ops = ["add", "mul", "exp", "sin", "relu"]
sizes = [1_000_000, 10_000_000, 50_000_000, 100_000_000]
size_labels = ["1M", "10M", "50M", "100M"]
results = []

for op in ops:
    for sz, sz_label in zip(sizes, size_labels):
        for dev_name, dev in [("ROCm", torch.device("cuda")), ("CPU", torch.device("cpu"))]:
            if dev_name == "ROCm" and not is_rocm_available():
                continue
            print(f"  {op:>4s} | {sz_label:>4s} | {dev_name} ...", end=" ", flush=True)
            r = benchmark_elementwise(num_elements=sz, op=op, device=dev, iterations=10)
            results.append({
                "Op": op,
                "Elements": sz_label,
                "Device": dev_name,
                "Time (ms)": r["mean_time_s"] * 1000,
            })
            print(f"{r['mean_time_s']*1000:.2f} ms")

df = pd.DataFrame(results)
print("\nElement-wise benchmark complete!")

## 3. Visualization — Per-operation Charts

In [ ]:
fig, axes = plt.subplots(1, len(ops), figsize=(20, 5), sharey=False)

for ax, op in zip(axes, ops):
    df_op = df[df["Op"] == op]
    x = range(len(sizes))
    width = 0.35

    rocm_vals = df_op[df_op["Device"] == "ROCm"]["Time (ms)"].tolist()
    cpu_vals = df_op[df_op["Device"] == "CPU"]["Time (ms)"].tolist()

    if rocm_vals:
        ax.bar([i - width / 2 for i in x], rocm_vals, width, label="ROCm", color="#E4002B")
    ax.bar([i + width / 2 for i in x], cpu_vals, width, label="CPU", color="#4A90D9")

    ax.set_xlabel("Elements")
    ax.set_ylabel("Time (ms)")
    ax.set_title(f"torch.{op}")
    ax.set_xticks(list(x))
    ax.set_xticklabels(size_labels, fontsize=8)
    ax.legend(fontsize=8)

plt.suptitle("Element-wise Operations: ROCm vs CPU", fontsize=14, y=1.02)
plt.tight_layout()
plt.show()

## 4. Speedup Heatmap

In [ ]:
if is_rocm_available():
    speedup_data = []
    for op in ops:
        row = {}
        for sz_label in size_labels:
            rocm_t = df[(df["Op"] == op) & (df["Elements"] == sz_label) & (df["Device"] == "ROCm")]["Time (ms)"].values[0]
            cpu_t = df[(df["Op"] == op) & (df["Elements"] == sz_label) & (df["Device"] == "CPU")]["Time (ms)"].values[0]
            row[sz_label] = cpu_t / rocm_t
        speedup_data.append(row)

    df_speedup = pd.DataFrame(speedup_data, index=ops)

    fig, ax = plt.subplots(figsize=(8, 4))
    sns.heatmap(
        df_speedup, annot=True, fmt=".1f", cmap="RdYlGn",
        center=1.0, linewidths=0.5, ax=ax,
        cbar_kws={"label": "Speedup (CPU time / ROCm time)"},
    )
    ax.set_title("ROCm Speedup over CPU")
    ax.set_xlabel("Tensor Size")
    ax.set_ylabel("Operation")
    plt.tight_layout()
    plt.show()
else:
    print("ROCm not available — cannot compute speedup heatmap.")

## 5. Speedup Summary Table

In [ ]:
if is_rocm_available():
    rows = []
    for op in ops:
        for sz_label in size_labels:
            rocm_t = df[(df["Op"] == op) & (df["Elements"] == sz_label) & (df["Device"] == "ROCm")]["Time (ms)"].values[0]
            cpu_t = df[(df["Op"] == op) & (df["Elements"] == sz_label) & (df["Device"] == "CPU")]["Time (ms)"].values[0]
            rows.append({
                "Op": op,
                "Elements": sz_label,
                "ROCm (ms)": f"{rocm_t:.2f}",
                "CPU (ms)": f"{cpu_t:.2f}",
                "Speedup": f"{cpu_t / rocm_t:.1f}x",
            })
    display(pd.DataFrame(rows))
else:
    print("ROCm not available — ran CPU only.")